In [1]:
import pandas as pd
from fastai.vision.all import *
import numpy as np


In [2]:
df=pd.read_csv(r"C:\Users\enesm\OneDrive\Masaüstü\My_Deep_Learning_Journey\FastAi\PaddyDoc\train.csv")
df.label.value_counts()

label
normal                      1764
blast                       1738
hispa                       1594
dead_heart                  1442
tungro                      1088
brown_spot                   965
downy_mildew                 620
bacterial_leaf_blight        479
bacterial_leaf_streak        380
bacterial_panicle_blight     337
Name: count, dtype: int64

In [3]:
trn_path = r"C:\Users\enesm\OneDrive\Masaüstü\My_Deep_Learning_Journey\FastAi\PaddyDoc\train_images\bacterial_panicle_blight"

In [7]:
def train(arch,size, item=Resize(480,method='squish'),accum=1,finetune=True, epochs=12):
    dls=ImageDataLoaders.from_folder(trn_path,valid_pct=0.2,item_tfms=item,
                                 batch_tfms=aug_transforms(size=size,min_scale=0.75),bs=64//accum)
    cbs=GradientAccumulation(64) if accum else[]
    learn= vision_learner(dls,arch,metrics=error_rate,cbs=cbs).to_fp16()
    if finetune:
        learn.fine_tune(epochs,0.01)
        return learn.tta(dl=dls.test_dl(tst_files))
    else:
        learn.unfreeze()
        learn.fit_one_cycle(epochs,0.01)

In [5]:
train('convnext_small_in22k',128,epochs=1,accum=1,finetune=False)

c:\Users\enesm\AppData\Local\Programs\Python\Python310\lib\site-packages\timm\models\_factory.py:114: UserWarning: Mapping deprecated model name convnext_small_in22k to current convnext_small.fb_in22k.
  model = create_fn(


epoch,train_loss,valid_loss,error_rate,time
0,0.000000,0.000000,0.000000,00:03


In [4]:
import pynvml

def report_gpu():
    pynvml.nvmlInit()
    device_count = pynvml.nvmlDeviceGetCount()

    for i in range(device_count):
        handle = pynvml.nvmlDeviceGetHandleByIndex(i)
        info = pynvml.nvmlDeviceGetMemoryInfo(handle)
        print(f"GPU {i} - Memory Used: {info.used / 1024**2:.2f} MB")

    pynvml.nvmlShutdown()





In [5]:
report_gpu()
import gc
gc.collect()
torch.cuda.empty_cache()


GPU 0 - Memory Used: 226.00 MB


In [16]:
train('convnext_small_in22k', 128, epochs=1, accum=2, finetune=False)
report_gpu()


epoch,train_loss,valid_loss,error_rate,time
0,0.000000,0.000000,0.000000,00:02


GPU 0 - Memory Used: 3566.71 MB


In [7]:
train('convnext_small_in22k', 128, epochs=1, accum=16, finetune=False)
report_gpu()


c:\Users\enesm\AppData\Local\Programs\Python\Python310\lib\site-packages\timm\models\_factory.py:114: UserWarning: Mapping deprecated model name convnext_small_in22k to current convnext_small.fb_in22k.
  model = create_fn(


epoch,train_loss,valid_loss,error_rate,time
0,0.000000,0.000000,0.000000,00:07


GPU 0 - Memory Used: 2622.71 MB


In [32]:
report_gpu()
torch.cuda.empty_cache()

GPU 0 - Memory Used: 4488.71 MB


In [8]:
train('convnext_large_in22k', 224, epochs=1, accum=4, finetune=False)
report_gpu()


c:\Users\enesm\AppData\Local\Programs\Python\Python310\lib\site-packages\timm\models\_factory.py:114: UserWarning: Mapping deprecated model name convnext_large_in22k to current convnext_large.fb_in22k.
  model = create_fn(


epoch,train_loss,valid_loss,error_rate,time
0,0.000000,0.000000,0.000000,01:06


GPU 0 - Memory Used: 8176.71 MB


In [ ]:
train('vit_large_patch16_224', 224, epochs=1, accum=2, finetune=False)
report_gpu()

In [34]:
res = 640,480

In [35]:
models = {
    'convnext_large_in22k': {
        (Resize(res), (320,224)),
    }, 'vit_large_patch16_224': {
        (Resize(480, method='squish'), 224),
        (Resize(res), 224),
    }, 'swinv2_large_window12_192_22k': {
        (Resize(480, method='squish'), 192),
        (Resize(res), 192),
    }, 'swin_large_patch4_window7_224': {
        (Resize(res), 224),
    }
}

In [49]:
trn_path=r"C:\Users\enesm\OneDrive\Masaüstü\My_Deep_Learning_Journey\FastAi\PaddyDoc\train_images"

In [50]:
tta_res = []

for arch,details in models.items():
    for item,size in details:
        print('---',arch)
        print(size)
        print(item.name)
        tta_res.append(train(arch, size, item=item, accum=2)) #, epochs=1))
        gc.collect()
        torch.cuda.empty_cache()

--- convnext_large_in22k
(320, 224)
Resize -- {'size': (480, 640), 'method': 'crop', 'pad_mode': 'reflection', 'resamples': (<Resampling.BILINEAR: 2>, <Resampling.NEAREST: 0>), 'p': 1.0}


c:\Users\enesm\AppData\Local\Programs\Python\Python310\lib\site-packages\timm\models\_factory.py:114: UserWarning: Mapping deprecated model name convnext_large_in22k to current convnext_large.fb_in22k.
  model = create_fn(


epoch,train_loss,valid_loss,error_rate,time


KeyboardInterrupt: 

: 